# Analisando dataset de JobHunAI: `SDE-Nov21.csv`
Link do repo público: https://github.com/coral-lab-asu/job-hunt-AI/tree/main/jobs

In [22]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [23]:
import pandas as pd

df = pd.read_csv('SDE-Nov21.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1907 entries, 0 to 1906
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Position Title    1907 non-null   object
 1   Date              1907 non-null   object
 2   Apply             1907 non-null   object
 3   Work Model        1907 non-null   object
 4   Location          1907 non-null   object
 5   Company           1907 non-null   object
 6   Company Size      1833 non-null   object
 7   Company Industry  1672 non-null   object
 8   Salary            1860 non-null   object
 9   Qualifications    1900 non-null   object
 10  H1b Sponsored     1907 non-null   object
 11  Is New Grad       320 non-null    object
dtypes: object(12)
memory usage: 178.9+ KB


In [24]:
df.head()

,Position Title,Date,Apply,Work Model,Location,Company,Company Size,Company Industry,Salary,Qualifications,H1b Sponsored,Is New Grad
0,Software Engineer I,2025-11-21,https://jobright.ai/jobs/info/6920a799a49a885a...,Hybrid,"Colorado Springs, Colorado",KBR Careers,1001-5000,NaN,$75915-$113820 /yr,1. Bachelor’s Degree or Higher in Computer Sci...,not sure,yes
1,Electronic Security Systems Operator,2025-11-21,https://jobright.ai/jobs/info/6903e36963502c0a...,On Site,"Multi Location\nSweetwater, TX\nWaco, TX\nBrow...",Texas State Technical College,1001-5000,"Education,Information Technology","$69,050.88-$69,050.88 /yr",1. Associate’s degree or higher in Criminal Ju...,not sure,NaN
2,Sage Intacct Implementation Support Consultant,2025-11-21,https://jobright.ai/jobs/info/6920a69da49a885a...,Remote,Remote (United States),Net at Work,51-100,"Consulting,CRM",$75000-$85000 /yr,1. Candidates have worked in a role supporting...,not sure,NaN
3,Associate Technical Account Manager,2025-11-21,https://jobright.ai/jobs/info/6920a66ba49a885a...,Hybrid,"New York, NY",Yardi,10000+,"Property Management,PropTech",$31.25-$32.81 /hr,"1. Bachelor's Degree in Business, Finance, Mat...",not sure,NaN
4,Identity Governance Administration Systems Ana...,2025-11-21,https://jobright.ai/jobs/info/69105ba5183be660...,On Site,"Charlotte, NC",Pacific Life,1001-5000,"Finance,Financial Services",$100620-$122980 /yr,"1. 1-3 years Information Security experience, ...",not sure,NaN


## Dados disponíveis

Seguindo os prints anteriores, notamos que temos 12 colunas:
* `Position Title`: Título
* `Date`: Data do anúncio
* `Apply`: URL
* `Work Model`: Modelo de trabalho (Híbrido, Remoto ou Presencial)
* `Location`: Localização
* `Company`: Empresa
* `Company Size`: Tamanho da empresa, em quantidade de funcionários
* `Company Industry`: Setor de Atuação (Finanças, Tecnologia, CRM, etc.)
* `Salary`: Salário
* `Qualifications`: Qualificações exigidas em formato de descrição de vaga
* `H1b Sponsored`: Se a empresa vai custear visto de trabalho (H1b é o visto de trabalho americano)
* `Is New Grad`: Se é vaga para recém-graduados:
    * 'yes': Sim
    * NaN: Não

## Separando vagas para recém-graduados e para mais experientes

In [25]:
print(f"Vagas para recém-graduados: {df['Is New Grad'].value_counts()['yes']}")
print(f"Vagas que exigem mais experiência: {df['Is New Grad'].isna().sum()}")

Vagas para recém-graduados: 320
Vagas que exigem mais experiência: 1587


In [26]:
df_new_grad = df.dropna(subset=['Is New Grad'])
df_seniors = df.loc[df['Is New Grad'].isna(),:]
df_new_grad.info()

<class 'pandas.core.frame.DataFrame'>
Index: 320 entries, 0 to 1902
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Position Title    320 non-null    object
 1   Date              320 non-null    object
 2   Apply             320 non-null    object
 3   Work Model        320 non-null    object
 4   Location          320 non-null    object
 5   Company           320 non-null    object
 6   Company Size      310 non-null    object
 7   Company Industry  291 non-null    object
 8   Salary            316 non-null    object
 9   Qualifications    320 non-null    object
 10  H1b Sponsored     320 non-null    object
 11  Is New Grad       320 non-null    object
dtypes: object(12)
memory usage: 32.5+ KB


In [27]:
df_seniors.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1587 entries, 1 to 1906
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Position Title    1587 non-null   object
 1   Date              1587 non-null   object
 2   Apply             1587 non-null   object
 3   Work Model        1587 non-null   object
 4   Location          1587 non-null   object
 5   Company           1587 non-null   object
 6   Company Size      1523 non-null   object
 7   Company Industry  1381 non-null   object
 8   Salary            1544 non-null   object
 9   Qualifications    1580 non-null   object
 10  H1b Sponsored     1587 non-null   object
 11  Is New Grad       0 non-null      object
dtypes: object(12)
memory usage: 161.2+ KB


## Ajuste do dataset para `loader-esco`

Em um dataset de oportunidades, o `loader-esco` espera encontrar as colunas `title` e `description`

Nesse dataset, o equivalente é:
* `title`: `Position Title`
* `description`: `Qualifications` 

In [28]:
df_new_grad = df_new_grad.rename(
    columns={
        'Position Title': 'title',
        'Qualifications': 'description'
    }
)

df_new_grad = df_new_grad[['title','description']]

df_seniors = df_seniors.rename(
    columns={
        'Position Title': 'title',
        'Qualifications': 'description'
    }
)
df_seniors = df_seniors[['title','description']]

df_new_grad.info()
print("\nContagem de nulos: \n", df_new_grad.isna().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 320 entries, 0 to 1902
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        320 non-null    object
 1   description  320 non-null    object
dtypes: object(2)
memory usage: 7.5+ KB

Contagem de nulos: 
 title          0
description    0
dtype: int64


In [29]:
df_seniors.info()
print("\nContagem de nulos: \n", df_seniors.isna().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 1587 entries, 1 to 1906
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        1587 non-null   object
 1   description  1580 non-null   object
dtypes: object(2)
memory usage: 37.2+ KB

Contagem de nulos: 
 title          0
description    7
dtype: int64


## Investigando e removando dados nulos em descrições

Ao olhar o dataset na coluna de descrições, nota-se que o de novos graduandos está completo, mas o de mais experientes não.

Vamos exibir e remover esses dados

In [30]:
print(df_seniors.loc[df_seniors['description'].isna(),:].head(10))

                                             title description
166                          Testing - Job posting         NaN
178   AI Trainer Onboarding and Support Specialist         NaN
313                    Technology Strategy Analyst         NaN
1334                  Technology Support Associate         NaN
1357                  Technology Support Associate         NaN
1489                     Site Reliability Engineer         NaN
1799                         IT Site Administrator         NaN


In [31]:
df_seniors = df_seniors.dropna()
df_seniors.info()
print("\nContagem de nulos: \n", df_seniors.isna().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 1580 entries, 1 to 1906
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        1580 non-null   object
 1   description  1580 non-null   object
dtypes: object(2)
memory usage: 37.0+ KB

Contagem de nulos: 
 title          0
description    0
dtype: int64


## Exportando para `.csv`

Agora que finalizamos o tratamento, vamos exportar para `.csv` para que o loader possa trabalhar com esses dados

In [32]:
df_new_grad.to_csv(path_or_buf="new_grad_dataset.csv",sep=",",)
df_seniors.to_csv(path_or_buf="seniors_dataset.csv",sep=",")

## Etapa 2. Unir datasets em um único

Notou-se que seniors e júniors não estavam bem separados pela base original (ex: Vaga Pós-Doutor em `new_grad.csv`, ou Vaga Júnior/Assitente em `seniors.csv`). Dessa forma, vamos voltar a analisar a base como um todo e apenas filtrar esses termos

In [33]:
df_new_grad_pt = pd.read_csv("new_grad_dataset_pt.csv")
df_seniors_pt = pd.read_csv("seniors_dataset_pt.csv")

In [34]:
#Concatenando
df_pt = pd.concat([df_new_grad_pt,df_seniors_pt],ignore_index=True)
df_pt.to_csv(path_or_buf="df_pt.csv",sep=",")

### 2.1. Procurando termos utilizados em título
Com base nos termos, poderemos descobrir o que pode ser utilizado para separar um sênior de um júnior

In [35]:
import re
import nltk

from nltk.corpus import stopwords

nltk.download("stopwords")

stop_words = set(stopwords.words("portuguese"))

tokens_unicos = set()

for title in df_pt["title"].dropna():
    tokens = re.findall(r"\b[\wÀ-ÿ]+\b", title.lower())

    tokens_filtrados = [
        token
        for token in tokens
        if token not in stop_words
    ]

    tokens_unicos.update(tokens_filtrados)

tokens_unicos = sorted(tokens_unicos)

for token in tokens_unicos:
    print(token)
print(f"Total de termos únicos: {len(tokens_unicos)}")

0
00044815
1
10
10111073
1042
10854
11
1279
1298
1396
141
1455
15
1519
1554
1555
1556
16
18
19
19h
2
20
2025
2026
2026a
21
21800
226
228q
23832
25
26
26129
2d
2ª
3
35
36000284
3d
4
4405
4499
5
53650
5887041
5969272
6
6004957
6004958
6014788
6014851
6014852
6014862
6014863
6014928
6021431
6055
734
7454
7h
90
9546
_ts
abastecimento
aberto
abril
abs
academia
aceitação
acelerada
acelerado
acesso
ad
adc
adc2s
administrador
administration
administrativos
administrator
administração
afton
agente
agentes
agricultura
agência
ai
aiops
airtable
alc
alemão
algoritmo
algoritmos
aliança
alta
amazon
ambientes
ameaças
aml
amz25652
amz26216
américa
analisador
analista
analistas
analytics
analítica
and
andamento
android
angeles
animador
anomalias
anos
anotação
antecipada
antes
anywhere
análise
análises
anúncios
aoc
apex
api
aplicada
aplicado
aplicativo
aplicativos
aplicação
aplicações
apoio
apostas
app
appsw
aprendizado
aprendizagem
aprovado
aquisição
arcgis
argila
aria
armazém
arquiteto
arquitetura
art

[nltk_data] Downloading package stopwords to /home/lsarai/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [36]:
#Selecionei termos com ChatGPT
termos_seniors = [
    "sênior", "senior", "líder", "lead", "staff", "principal",
    "diretor", "diretora", "gerente", "head", "chief", "cio",
    "cto", "cfo","ceo","coo","presidente", "vice-presidente",
    "doutor", "doutorado", "phd", "doutoral", "mestre", "mestrado",
    "especialista", "especialistas", "especialização", "executiva", "executivo",
    "fundador", "gestão","gestor"
]

#Seguindo esse segundo grupo, temos termos que indicam baixa senioridade
baixa_experiencia = [
    "júnior",
    "estagiário",
    "estágio",
    "bacharelado",
    "universitário"
]


In [37]:
pattern_senior = r"\b(?:" + "|".join(map(re.escape, termos_seniors)) + r")\b"
pattern_baixa_experiencia = r"\b(?:" + "|".join(map(re.escape, baixa_experiencia)) + r")\b"

mask_senior = df_pt["title"].str.contains(
    pattern_senior,
    case=False,
    na=False,
    regex=True
)

mask_baixa_experiencia = df_pt["title"].str.contains(
    pattern_baixa_experiencia,
    case=False,
    na=False,
    regex=True
)

df_senioridade = df_pt[
    mask_senior & ~mask_baixa_experiencia
].copy()

for title in sorted(df_senioridade["title"]):
    print(title)

print(f"Total: {len(df_senioridade['title'])}")

(Contratação Geral) Cientista de Pesquisa, Graduado (Recomendação do TikTok - Próxima Geração de Recomendações) - Início em 2026 (Doutorado)
(Contratação Geral) Engenheiro de Aprendizado de Máquina, Data-Search - São José - Início em 2026 (Doutorado)
1455 - Especialista em Segurança da Informação
Analista Sênior, Ciência de Dados
Analista Técnico Associado - Especialista em Suporte de TI
Analista de Gestão de Acesso
Analista de Negócios / Gerente de Projetos, Plataformas Bancárias
Analista de Sistemas e Desenvolvimento (nível 1, 2, 3 ou sênior)
Analista de Tecnologia do Departamento – Gestão de Equipamentos e Frota (Serviço Público)
Assistente e Gerente de Projetos
Assistente/Gerente de Dados e Sistemas
Associado Sênior, Ciência de Dados
Bolsa de pós-doutorado: Segurança do Paciente em Escala Global
Cientista de Aprendizado de Máquina (Graduação) – Recomendação de Conteúdo para E-commerce Global – Início em 2026 (Doutorado)
Cientista de Pesquisa - Pós-graduação em Recomendações de Cont

In [38]:
df_pt = df_pt.drop(df_senioridade.index).reset_index(drop=True)
print(f'Total restante: {len(df_pt)}')

Total restante: 1548


### 2.2. Limpando pela descrição

Algumas bases americanas possuem exigências de cidadania dos EUA, o que pode incomodar os usuários.

Sendo assim, vamos procurar o item que exige cidadania, removê-lo e reenumerar os requisitos

Ex:
```txt
1. *Apenas candidatos com cidadania local ou dos EUA.*
2. Capacidade de trabalhar de forma independente ou em pequenos grupos, demonstrando forte colaboração em equipe.
3. Habilidades eficazes de escuta e comunicação, especialmente em ambientes focados no suporte ao cliente.
4. Domínio do uso do teclado, gramática e ferramentas de comunicação interna para garantir a documentação e a comunicação claras.
5. Forte proficiência no uso de produtos Microsoft, incluindo Word, Excel e MS 365.
```

Após limpeza:
```txt
1. Capacidade de trabalhar de forma independente ou em pequenos grupos, demonstrando forte colaboração em equipe.
2. Habilidades eficazes de escuta e comunicação, especialmente em ambientes focados no suporte ao cliente.
3. Domínio do uso do teclado, gramática e ferramentas de comunicação interna para garantir a documentação e a comunicação claras.
4. Forte proficiência no uso de produtos Microsoft, incluindo Word, Excel e MS 365.
```


In [42]:
termos_cidadania = [
    "cidadania",
    "cidadão",
    "cidadãos",
    "cidadã",
    "cidadãs"
]

pattern_cidadania = r"\b(?:" + "|".join(map(re.escape, termos_cidadania)) + r")\b"

mask_cidadania = df_pt["description"].str.contains(
    pattern_cidadania,
    case=False,
    na=False,
    regex=True
)

df_cidadania = df_pt[mask_cidadania].copy()

for _,title,desc in df_cidadania.values:
    print(title,desc,sep="\n",end="\n\n")

print(f"Total: {len(df_cidadania['description'])}")

Engenheiro de Campo Júnior
1. Ensino Médio completo (0-2 anos de experiência ou equivalente)
2. Cidadão dos EUA
3. Alto nível de proficiência técnica
4. Excelentes habilidades de resolução de problemas e capacidade analítica
5. Deve ser capaz de obter e manter a aprovação de segurança para funcionários do governo

Engenheiro de Inteligência Artificial
1. Formação: Bacharelado em Artes/Bacharelado em Ciências
2. Experiência: +0 anos de experiência relevante
3. Habilidades técnicas: Ciência de Dados, R, Python, LLMs
4. Nível de segurança: deve ser capaz de obter o nível 4 de Confiança Pública da EPA
5. Necessário ser cidadão dos EUA

2026 – Engenheiro de Infraestrutura de Software (Nível I) – Presencial (Texas)
1. Bacharelado em Ciências, Tecnologia, Engenharia ou Matemática (STEM)
2. Experiência em desenvolvimento de software utilizando linguagens de script, como: Python, Perl, Ruby, Java, Bash ou Shell.
3. Esta posição exige experiência profissional relevante de 12 meses ou menos (excl

In [ ]:
# Removendo menção à cidadania e reenumerando os itens
df_cidadania["description"] = df_cidadania["description"].apply(
    lambda description: "\n".join(
        re.sub(r"^\s*\d+\.\s*", f"{i}. ", linha.strip())
        for i, linha in enumerate(
            [
                linha.strip()
                for linha in description.split("\n")
                if linha.strip()
                and not re.search(
                    r"\b(?:cidadania|cidadão|cidadãos|cidadã|cidadãs)\b",
                    linha,
                    flags=re.IGNORECASE
                )
            ],
            start=1
        )
    )
)

for desc in sorted(df_cidadania["description"]):
    print(desc, end="\n\n")

print(f"Total: {len(df_cidadania['description'])}")





1. Atualmente cursando o Bacharelado em Administração (ou equivalente) e com previsão de conclusão até dezembro de 2025 ou junho de 2026.
2. Forte capacidade analítica para resolver problemas e habilidade de aprendizado rápido.
3. Capacidade de gerenciar múltiplos projetos e equipes multidisciplinares com mínima supervisão.
4. Fortes habilidades de comunicação e trabalho em equipe.

1. Atualmente cursando o bacharelado em Engenharia, com previsão de conclusão entre dezembro de 2025 ou junho de 2026.
2. Forte capacidade analítica para resolver problemas e habilidade de aprendizado rápido.
3. Capacidade de gerenciar múltiplos projetos e equipes multidisciplinares com mínima supervisão.
4. Fortes habilidades de comunicação e trabalho em equipe.

1. Atualmente cursando o bacharelado em Engenharia, com previsão de conclusão entre dezembro de 2025 ou junho de 2026.
2. Forte capacidade analítica para resolver problemas e habilidade de aprendizado rápido.
3. Capacidade de gerenciar múltipl

In [ ]:
df_pt.loc[df_cidadania.index, "description"] = df_cidadania["description"]

mask_cidadania = df_pt["description"].str.contains(
    pattern_cidadania,
    case=False,
    na=False,
    regex=True
)

df_cidadania_validacao = df_pt[mask_cidadania].copy()

for desc in sorted(df_cidadania_validacao["description"]):
    print(desc,end="\n\n")
#Deve exibir 0, pois o termo já foi excluído
print(f"Total: {len(df_cidadania_validacao['description'])}")

Total: 0


In [ ]:
df_pt.to_csv(path_or_buf="new_grad_pt_final.csv",sep=",")